## RAG Implementation using OpenSearch and KServe

In [ ]:
import requests

In [ ]:
import os

Read environment variables

In [ ]:
os_endpoint = f"https://{os.environ['OPENSEARCH_ENDPOINTS']}"
os_username=f"{os.environ['OPENSEARCH_USERNAME']}"
os_password=f"{os.environ['OPENSEARCH_PASSWORD']}"
os_certificate=f"{os.environ['OPENSEARCH_CA_CERT_PATH']}"

In [ ]:
os_endpoint

In [ ]:
os_username

In [ ]:
os_certificate

In [ ]:
base_url = f"{os_endpoint.split(',')[0].rstrip('/')}"

print(base_url)

def make_request(url, method, payload):
    _method = method.lower()
    if _method == "post":
        f = requests.post
    elif _method == "get":
        f = requests.get
    elif _method == "put":
        f = requests.put
    else:
        raise ValueError(f"method {method} not supported")

    _url = f"{base_url}/{url}"

    headers = {
        "Content-Type": "application/json"
    }
    
    params = {
        "auth": (os_username, os_password),
        "headers": headers,
        "verify": os_certificate
    } | ({"json": payload} if payload else {})

    response = f(
        _url, **params
    )
    # 5. Print the results
    print(f"Status Code: {response.status_code}")
    
    json_response = response.json()
    print("Response JSON:", json_response)
    
    return json_response

## Create Indexed Knowledge base of Documents

Provide the name of the index collection

In [ ]:
index_name = "news"

### Register and Deploy Embedding Model

In [ ]:
model_group_id = make_request(
    "_plugins/_ml/model_groups/_register",
    "POST",
    {
        "name": "rag-model-group-1",
        "description": "A model group for RAG use case models"
    }
)["model_group_id"]

In [ ]:
model_group_id

In [ ]:
task_id = make_request(
    "_plugins/_ml/models/_register",
    "POST",
    {
        "name": "huggingface/sentence-transformers/msmarco-distilbert-base-tas-b",
        "version": "1.0.2",
        "model_group_id": model_group_id,
        "model_format": "TORCH_SCRIPT"
    }
)["task_id"]

In [ ]:
task_id

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential

In [ ]:
@retry(
    wait=wait_exponential(multiplier=2, min=1, max=10),
    stop=stop_after_attempt(10),
    reraise=True,
)
def get_model_id(task_id):
    json_response = make_request(
        f"_plugins/_ml/tasks/{task_id}",
        "GET",
        {}
    )
    assert json_response["state"] == "COMPLETED"
    return json_response["model_id"]

In [ ]:
model_id = get_model_id(task_id)

In [ ]:
model_id

In [ ]:
make_request(
    f"_plugins/_ml/models/{model_id}/_deploy",
    "POST",
    {}
)

Test the embedding model

In [ ]:
inference_output = make_request(
    f"_plugins/_ml/_predict/text_embedding/{model_id}",
    "POST",
    {
        "text_docs":[ "What is Ubuntu Pro?"],
        "return_number": "true",
        "target_response": ["sentence_embedding"]
    }
)

### Ingested indexed collection of documents

In [ ]:
make_request(
    f"_ingest/pipeline/rag-ingest-news-pipeline",
    "PUT", 
    {
        "description": "An RAG ingest pipeline",
        "processors": [{
            "text_embedding": {
                "model_id": model_id,
                "field_map": {
                    "text": "sentence_embedding"
                }
            }
        }]
    }
)

In [ ]:
make_request(
    index_name,
    "PUT",
    {
        "settings": {
            "index.knn": "true",
            "default_pipeline": "rag-ingest-news-pipeline"
        },
        "mappings": {
            "properties": {
                "id": {"type": "text"},
                "sentence_embedding": {
                    "type": "knn_vector",
                    "dimension": 768,
                    "method": {
                        "name": "hnsw",
                        "engine": "faiss",
                        "space_type": "l2",
                        "parameters": {
                            "encoder": {
                                "name": "sq",
                                "parameters": {"type": "fp16"}
                            },
                            "ef_construction": 256,
                            "m": 8
                        }
                    }
                },
                "text": {"type": "text"}
            }
        }
    }
)

## Import the dataset

Download the dataset using

`curl -L -o news-category-dataset.zip  https://www.kaggle.com/api/v1/datasets/download/rmisra/news-category-dataset`

and extract it locally

In [2]:
import json

In [ ]:
with open("news/News_Category_Dataset_v3.json", "r") as fid:
    data = [json.loads(row) for row in fid.readlines()]

In [ ]:
documents = [f"Title: {d['headline']} - {d['short_description']}" for d in data[:100]] 

In [ ]:
documents[0]

In [ ]:
for idoc, doc in enumerate(documents):
    print(f"Adding doc #{idoc}")
    make_request(
        f"{index_name}/_doc",
        "POST",
        {
            "text": doc
        }
    )

Test a query

In [ ]:
make_request(
    f"{index_name}/_search",
    "GET",
    {
        "_source": {
            "excludes": [
                "sentence_embedding"
            ]
        },
        "query": {
            "neural": {
                "sentence_embedding": {
                    "query_text": "Do you have information about covid boosters?",
                    "model_id": model_id,
                    "k": 5
                }
            }
        }
    }
)

## Integrate the LLM

First, define the model and allow connection

In [ ]:
model_endpoint="10.152.183.189/openai"
model_name="qwen25-3b-native"

In [ ]:
make_request(
    "_cluster/settings", 
    "PUT",
    {
        "persistent": {
            "plugins.ml_commons.connector.private_ip_enabled": True,
            "plugins.ml_commons.trusted_connector_endpoints_regex": [
                "^http://.*$"
            ]
        }
    }
)

### Create the connections to the external LLM

In [ ]:
connector_id = make_request(
    "_plugins/_ml/connectors/_create",
    "POST",
    {
        "name": "vLLM Connector",
        "description": "OpenAI-compatible vLLM API",
        "version": 2,
        "protocol": "http",
        "parameters": {
            "endpoint": model_endpoint,
            "model": model_name,
            "temperature": 0.03,
            "repetition_penalty": 1.13
        },
        "credential": {
            "api_key": "not-needed-for-local-vllm"
        },
        "actions": [{
            "action_type": "predict",
            "method": "POST",
            "url": "http://${parameters.endpoint}/v1/chat/completions",
            "request_body": "{ \"model\": \"${parameters.model}\", \"messages\": ${parameters.prompt}, \"temperature\":${parameters.temperature}, \"repetition_penalty\": ${parameters.repetition_penalty} }"
        }]
    }
)["connector_id"]

In [ ]:
task_id_llm = make_request(
    "_plugins/_ml/models/_register",
    "POST",
    {
        "name": f"KServe Inference Service: {model_name}",
        "function_name": "remote",
        "description": "RAG LLM test model",
        "connector_id": connector_id
    }
)["task_id"]

In [ ]:
model_id_llm = get_model_id(task_id_llm)

In [ ]:
model_id_llm

In [ ]:
make_request(
    f"_plugins/_ml/models/{model_id_llm}/_deploy",
    "POST",
    {}
)

### Define Conversational Agent

In [ ]:
agent_id = make_request(
    "_plugins/_ml/agents/_register",
    "POST",
    {
        "name": "test-agent-vector-mlmodel-rag",
        "type": "conversational_flow",
        "description": "This is a demo agent for conversational flow",
        "app_type": "rag",
        "memory": {
            "type": "conversation_index"
        },
        "tools": [
            {
                "type": "VectorDBTool",
                "parameters": {
                    "model_id": model_id,
                    "index": index_name,
                    "embedding_field": "sentence_embedding",
                    "source_field": ["text"],
                    "input": "${parameters.question}",
                    "doc_size": 10,
                    "k": 30
                }
            },
            {
                "type": "MLModelTool",
                "description": "A RAG LLM tool to answer questions",
                "parameters": {
                    "model_id": model_id_llm,
                    "prompt": '[ { "role": "system", "content": "${parameters.VectorDBTool.output}" }, { "role": "system", "content": "${parameters.chat_history:-}" }, { "role": "user", "content": "${parameters.question}" } ]'
                }
            }
        ]
    }
)["agent_id"]

In [ ]:
agent_id

Test out the agent

In [ ]:
inference_output = make_request(
    f"_plugins/_ml/agents/{agent_id}/_execute",
    "POST",
    {
        "parameters": {
            "question": "What can you tell me about Covid booster shots?"
        }
    }
)

In [ ]:
memory_id = [output["result"] for output in inference_output["inference_results"][0]["output"] if output["name"]=="memory_id"][0]

In [ ]:
memory_id

In [ ]:
inference_output = make_request(
    f"_plugins/_ml/agents/{agent_id}/_execute",
    "POST",
    {
        "parameters": {
            "question": "Do you have other information about health and diseases?",
            "memory_id": memory_id,
            "message_history_limit": 10
        }
    }
)